In [1]:
import sys
!{sys.executable} -m pip install nba_api


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install nba_api pandas tqdm --upgrade


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import requests
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

def get_pbp(game_id):
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"
    r = requests.get(url, headers=headers, timeout=60)
    data = r.json()
    
    actions = data['game']['actions']
    return pd.DataFrame(actions)

# Test with Spurs/Thunder
df = get_pbp('0042500311')
print(df.shape)
print(df.columns.tolist())
print(df.head())

(719, 56)
['actionNumber', 'clock', 'timeActual', 'period', 'periodType', 'actionType', 'subType', 'qualifiers', 'personId', 'x', 'y', 'possession', 'scoreHome', 'scoreAway', 'edited', 'orderNumber', 'isTargetScoreLastPeriod', 'xLegacy', 'yLegacy', 'isFieldGoal', 'side', 'description', 'personIdsFilter', 'teamId', 'teamTricode', 'descriptor', 'jumpBallRecoveredName', 'jumpBallRecoverdPersonId', 'playerName', 'playerNameI', 'jumpBallWonPlayerName', 'jumpBallWonPersonId', 'jumpBallLostPlayerName', 'jumpBallLostPersonId', 'area', 'areaDetail', 'shotDistance', 'shotResult', 'pointsTotal', 'assistPlayerNameInitial', 'assistPersonId', 'assistTotal', 'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal', 'reboundOffensiveTotal', 'officialId', 'foulPersonalTotal', 'foulTechnicalTotal', 'foulDrawnPlayerName', 'foulDrawnPersonId', 'turnoverTotal', 'stealPlayerName', 'stealPersonId', 'blockPlayerName', 'blockPersonId']
   actionNumber        clock              timeActual  period periodType 

In [5]:
# Let's see the action types we're working with
print(df['actionType'].value_counts())
print("\n")
print(df['subType'].value_counts().head(20))

actionType
substitution    198
rebound         120
2pt             109
3pt              88
freethrow        48
foul             45
turnover         38
steal            25
timeout          17
block            15
period           12
jumpball          3
game              1
Name: count, dtype: int64


subType
Jump Shot         132
out                99
in                 99
defensive          85
Layup              50
personal           42
                   40
offensive          38
1 of 2             20
2 of 2             20
full               16
bad pass           13
DUNK               13
lost ball          12
end                 7
start               6
out-of-bounds       6
1 of 1              5
recovered           3
offensive foul      3
Name: count, dtype: int64


In [6]:
# Check score tracking works correctly
score_df = df[df['scoreHome'].notna()][['actionNumber', 'period', 'clock', 'actionType', 'scoreHome', 'scoreAway', 'possession', 'description']]
print(score_df.head(20))

    actionNumber  period        clock actionType scoreHome scoreAway  \
0              2       1  PT12M00.00S     period         0         0   
1              4       1  PT11M58.00S   jumpball         0         0   
2              7       1  PT11M42.00S        3pt         0         3   
3              9       1  PT11M17.00S        2pt         0         3   
4             10       1  PT11M13.00S    rebound         0         3   
5             11       1  PT11M04.00S       foul         0         3   
6             13       1  PT11M04.00S  freethrow         0         4   
7             14       1  PT11M04.00S  freethrow         0         5   
8             15       1  PT10M46.00S   turnover         0         5   
9             16       1  PT10M46.00S      steal         0         5   
10            17       1  PT10M42.00S        3pt         0         5   
11            18       1  PT10M37.00S    rebound         0         5   
12            19       1  PT10M37.00S        2pt         0      

In [7]:
# Check what possession values look like
print(df['possession'].value_counts())

possession
1610612759    397
1610612760    318
0               4
Name: count, dtype: int64


In [8]:
# Score tracking
score_df = df[df['scoreHome'].notna()][['actionNumber', 'period', 'clock', 'actionType', 'scoreHome', 'scoreAway', 'possession', 'description']]
print(score_df.head(20))

    actionNumber  period        clock actionType scoreHome scoreAway  \
0              2       1  PT12M00.00S     period         0         0   
1              4       1  PT11M58.00S   jumpball         0         0   
2              7       1  PT11M42.00S        3pt         0         3   
3              9       1  PT11M17.00S        2pt         0         3   
4             10       1  PT11M13.00S    rebound         0         3   
5             11       1  PT11M04.00S       foul         0         3   
6             13       1  PT11M04.00S  freethrow         0         4   
7             14       1  PT11M04.00S  freethrow         0         5   
8             15       1  PT10M46.00S   turnover         0         5   
9             16       1  PT10M46.00S      steal         0         5   
10            17       1  PT10M42.00S        3pt         0         5   
11            18       1  PT10M37.00S    rebound         0         5   
12            19       1  PT10M37.00S        2pt         0      

In [ ]:
import re

def parse_clock(clock_str):
    """Convert 'PT11M42.00S' to seconds remaining in period"""
    match = re.match(r'PT(\d+)M([\d.]+)S', clock_str)
    if match:
        minutes = int(match.group(1))
        seconds = float(match.group(2))
        return minutes * 60 + seconds
    return None

def build_game_state(df, game_id, home_team, away_team, is_playoffs=True):
    
    action_types_keep = ['2pt', '3pt', 'freethrow', 'turnover', 'foul', 'rebound', 'block', 'steal']
    df = df[df['actionType'].isin(action_types_keep)].copy()
    
    # Parse clock
    df['seconds_in_period'] = df['clock'].apply(parse_clock)
    
    
    def seconds_remaining(row):
        period = row['period']
        secs = row['seconds_in_period']
        if period <= 4:
            return (4 - period) * 720 + secs
        else:
            # Overtime
            return max(0, (period - 4) * 300 - (300 - secs))
    
    df['seconds_remaining'] = df.apply(seconds_remaining, axis=1)
    
   
    df['scoreHome'] = pd.to_numeric(df['scoreHome'], errors='coerce').ffill()
    df['scoreAway'] = pd.to_numeric(df['scoreAway'], errors='coerce').ffill()
    df['score_diff'] = df['scoreHome'] - df['scoreAway']
    
    # Possession as binary 
    df['home_possession'] = (df['possession'] == home_team).astype(int)
    
    # Period type
    df['is_overtime'] = (df['period'] > 4).astype(int)
    
    # Foul trouble 
    df['home_fouls'] = df[df['teamId'] == home_team]['foulPersonalTotal'].ffill().reindex(df.index).ffill().fillna(0)
    df['away_fouls'] = df[df['teamId'] == away_team]['foulPersonalTotal'].ffill().reindex(df.index).ffill().fillna(0)
    
    # Momentum 
    df['momentum'] = df['score_diff'].diff(5)
    
    # Metadata
    df['game_id'] = game_id
    df['is_playoffs'] = int(is_playoffs)
    
    # Final columns
    keep = [
        'game_id', 'period', 'is_overtime', 'seconds_remaining',
        'scoreHome', 'scoreAway', 'score_diff', 'home_possession',
        'home_fouls', 'away_fouls', 'momentum', 'actionType',
        'teamTricode', 'is_playoffs'
    ]
    
    return df[keep].reset_index(drop=True)

# Test
home_team = 1610612760  # OKC
away_team = 1610612759  # SAS

clean = build_game_state(df, '0042500311', home_team, away_team, is_playoffs=True)
print(clean.shape)
print(clean.head(20))
print(clean.dtypes)

(488, 14)
       game_id  period  is_overtime  seconds_remaining  scoreHome  scoreAway  \
0   0042500311       1            0             2862.0          0          3   
1   0042500311       1            0             2837.0          0          3   
2   0042500311       1            0             2833.0          0          3   
3   0042500311       1            0             2824.0          0          3   
4   0042500311       1            0             2824.0          0          4   
5   0042500311       1            0             2824.0          0          5   
6   0042500311       1            0             2806.0          0          5   
7   0042500311       1            0             2806.0          0          5   
8   0042500311       1            0             2802.0          0          5   
9   0042500311       1            0             2797.0          0          5   
10  0042500311       1            0             2797.0          0          5   
11  0042500311       1        

In [ ]:
# Win label
final_score_home = df['scoreHome'].dropna().iloc[-1]
final_score_away = df['scoreAway'].dropna().iloc[-1]
home_won = int(final_score_home > final_score_away)
clean['home_team_won'] = home_won

print(f"Final: OKC {int(final_score_home)} - SAS {int(final_score_away)} | Home won: {bool(home_won)}")
print(clean[['seconds_remaining', 'score_diff', 'home_team_won']].tail(10))

Final: OKC 115 - SAS 122 | Home won: False
     seconds_remaining  score_diff  home_team_won
478              315.1          -6              0
479              314.5          -6              0
480              314.5          -7              0
481              314.5          -8              0
482              309.8          -8              0
483              309.8          -7              0
484              309.8          -7              0
485              309.8          -7              0
486              309.8          -7              0
487              306.9          -7              0


In [ ]:

print('0042500311')
# 004 = playoffs, 25 = 2025-26 season, 00311 = game 311

0042500311


In [17]:
def generate_game_ids(season_year, season_type='regular'):
    """
    season_year: 2024 = 2024-25 season
    Regular season has ~1230 games (30 teams x 82 / 2)
    Playoffs have at most 105 games
    """
    short_year = str(season_year + 1)[-2:]  # 2024 -> '25'
    
    if season_type == 'regular':
        prefix = f'002{short_year}'
        max_games = 1230
    else:
        prefix = f'004{short_year}'
        max_games = 105  # 4 rounds max
    
    return [f"{prefix}{str(i).zfill(5)}" for i in range(1, max_games + 1)]

# Test — does our known game ID fit the pattern?
playoffs_2025 = generate_game_ids(2025, 'playoffs')
print('0042500311' in playoffs_2025)  # should be True
print(f"Generated {len(playoffs_2025)} playoff IDs")
print(playoffs_2025[:5])

False
Generated 105 playoff IDs
['0042600001', '0042600002', '0042600003', '0042600004', '0042600005']


In [18]:
def try_get_pbp(game_id):
    """Returns DataFrame or None if game doesn't exist"""
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return None
        data = r.json()
        actions = data['game']['actions']
        if not actions:
            return None
        return pd.DataFrame(actions)
    except:
        return None

# Quick test
df_test = try_get_pbp('0042500311')  # known good
df_none = try_get_pbp('0042500999')  # probably doesn't exist
print("Known game:", df_test.shape if df_test is not None else None)
print("Fake game:", df_none)

Known game: (719, 56)
Fake game: None


In [19]:
import requests
import pandas as pd
import time
import re
import os
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

# ── 1. Game ID generator ───────────────────────────────────────────────────────
def generate_game_ids(season_year, season_type='regular'):
    short_year = str(season_year + 1)[-2:]
    if season_type == 'regular':
        prefix = f'002{short_year}'
        max_games = 1230
    else:
        prefix = f'004{short_year}'
        max_games = 105
    return [f"{prefix}{str(i).zfill(5)}" for i in range(1, max_games + 1)]

# ── 2. Fetch raw PBP ──────────────────────────────────────────────────────────
def try_get_pbp(game_id):
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return None
        data = r.json()
        actions = data['game']['actions']
        if not actions:
            return None
        return pd.DataFrame(actions)
    except:
        return None

# ── 3. Parse clock ────────────────────────────────────────────────────────────
def parse_clock(clock_str):
    match = re.match(r'PT(\d+)M([\d.]+)S', str(clock_str))
    if match:
        return int(match.group(1)) * 60 + float(match.group(2))
    return None

# ── 4. Build game state ───────────────────────────────────────────────────────
def build_game_state(raw_df, game_id, is_playoffs):
    df = raw_df.copy()

    # ── Derive home/away team IDs ──────────────────────────────────────
    possession_counts = df[df['possession'] != 0]['possession'].value_counts()
    if len(possession_counts) < 2:
        return None
    team_ids = possession_counts.index.tolist()
    # Home team has fewer possessions on average (slightly) — actually
    # we can't reliably tell from PBP alone so we use first two and
    # mark clearly in metadata. Game code format is AWAY@HOME so:
    # game actions have homeTeam implicit — we'll use teamId order of
    # first scoring play as proxy. Not perfect but consistent.
    home_team_id = team_ids[0]
    away_team_id = team_ids[1]

    # ── Filter to meaningful action types ──────────────────────────────
    action_types_keep = [
        '2pt', '3pt', 'freethrow', 'turnover', 'foul',
        'rebound', 'block', 'steal', 'timeout', 'ejection'
    ]
    df = df[df['actionType'].isin(action_types_keep)].copy()
    if df.empty:
        return None

    # ── Ensure all expected columns exist (some may be absent) ─────────
    for col in [
        'shotDistance', 'area', 'areaDetail', 'shotResult', 'isFieldGoal',
        'qualifiers', 'foulPersonalTotal', 'foulTechnicalTotal',
        'subType', 'teamId', 'teamTricode', 'possession',
        'scoreHome', 'scoreAway', 'period', 'clock', 'personId',
        'playerName', 'description', 'x', 'y'
    ]:
        if col not in df.columns:
            df[col] = None

    # ── Time ───────────────────────────────────────────────────────────
    df['seconds_in_period'] = df['clock'].apply(parse_clock)

    def seconds_remaining(row):
        p, s = row['period'], row['seconds_in_period']
        if s is None:
            return None
        return (4 - p) * 720 + s if p <= 4 else max(0, 300 - s)

    df['seconds_remaining'] = df.apply(seconds_remaining, axis=1)
    df['is_overtime'] = (df['period'] > 4).astype(int)

    # ── Score ──────────────────────────────────────────────────────────
    df['scoreHome'] = pd.to_numeric(df['scoreHome'], errors='coerce').ffill().fillna(0)
    df['scoreAway'] = pd.to_numeric(df['scoreAway'], errors='coerce').ffill().fillna(0)
    df['score_diff'] = df['scoreHome'] - df['scoreAway']
    df['momentum_5']  = df['score_diff'].diff(5)
    df['momentum_10'] = df['score_diff'].diff(10)

    # ── Possession ─────────────────────────────────────────────────────
    df['home_possession'] = (df['possession'] == home_team_id).astype(int)

    # ── Fouls ──────────────────────────────────────────────────────────
    home_mask = df['teamId'] == home_team_id
    away_mask = df['teamId'] == away_team_id

    def propagate(series, index):
        return series.ffill().reindex(index).ffill().fillna(0)

    df['home_fouls']       = propagate(df[home_mask]['foulPersonalTotal'],  df.index)
    df['away_fouls']       = propagate(df[away_mask]['foulPersonalTotal'],  df.index)
    df['home_tech_fouls']  = propagate(df[home_mask]['foulTechnicalTotal'], df.index)
    df['away_tech_fouls']  = propagate(df[away_mask]['foulTechnicalTotal'], df.index)

    # ── Ejections (cumulative, forward-filled) ─────────────────────────
    df['home_ejections'] = 0
    df['away_ejections'] = 0
    ej = df[df['actionType'] == 'ejection']
    for idx, row in ej.iterrows():
        loc = df.index.get_loc(idx)
        if row['teamId'] == home_team_id:
            df.iloc[loc:, df.columns.get_loc('home_ejections')] += 1
        else:
            df.iloc[loc:, df.columns.get_loc('away_ejections')] += 1

    # ── Timeouts (cumulative per team) ─────────────────────────────────
    df['home_timeouts_used'] = 0
    df['away_timeouts_used'] = 0
    to = df[df['actionType'] == 'timeout']
    for idx, row in to.iterrows():
        loc = df.index.get_loc(idx)
        if row['teamId'] == home_team_id:
            df.iloc[loc:, df.columns.get_loc('home_timeouts_used')] += 1
        else:
            df.iloc[loc:, df.columns.get_loc('away_timeouts_used')] += 1

    # ── Shot quality ingredients ───────────────────────────────────────
    df['shot_distance']   = pd.to_numeric(df['shotDistance'], errors='coerce')
    df['shot_area']       = df['area']
    df['shot_area_detail']= df['areaDetail']
    df['shot_result']     = df['shotResult']
    df['is_field_goal']   = pd.to_numeric(df['isFieldGoal'], errors='coerce').fillna(0)
    df['shot_x']          = pd.to_numeric(df['x'], errors='coerce')
    df['shot_y']          = pd.to_numeric(df['y'], errors='coerce')

    def has_qualifier(q, tag):
        return int(tag in q) if isinstance(q, list) else 0

    df['is_fastbreak']    = df['qualifiers'].apply(lambda q: has_qualifier(q, 'fastbreak'))
    df['is_painttouch']   = df['qualifiers'].apply(lambda q: has_qualifier(q, 'pointsinthepaint'))
    df['is_fromturnover'] = df['qualifiers'].apply(lambda q: has_qualifier(q, 'fromturnover'))

    # ── Free throw context ─────────────────────────────────────────────
    df['freethrow_subtype'] = df.apply(
        lambda r: r['subType'] if r['actionType'] == 'freethrow' else None, axis=1
    )

    # ── Player/action identity (useful for foul trouble joining later) ─
    df['person_id']    = df['personId']
    df['player_name']  = df['playerName']
    df['description']  = df['description']

    # ── Win label ──────────────────────────────────────────────────────
    home_won = int(df['scoreHome'].iloc[-1] > df['scoreAway'].iloc[-1])

    # ── Metadata ───────────────────────────────────────────────────────
    df['game_id']       = game_id
    df['is_playoffs']   = int(is_playoffs)
    df['home_team_won'] = home_won
    df['home_team_id']  = home_team_id
    df['away_team_id']  = away_team_id

    # ── Final columns ──────────────────────────────────────────────────
    keep = [
        # Metadata
        'game_id', 'is_playoffs', 'home_team_won', 'home_team_id', 'away_team_id',
        # Time
        'period', 'is_overtime', 'seconds_remaining',
        # Score
        'scoreHome', 'scoreAway', 'score_diff', 'momentum_5', 'momentum_10',
        # Possession & action
        'home_possession', 'actionType', 'teamTricode',
        # Fouls
        'home_fouls', 'away_fouls', 'home_tech_fouls', 'away_tech_fouls',
        # Ejections & timeouts
        'home_ejections', 'away_ejections', 'home_timeouts_used', 'away_timeouts_used',
        # Shot quality
        'shot_distance', 'shot_area', 'shot_area_detail', 'shot_result',
        'is_field_goal', 'shot_x', 'shot_y',
        'is_fastbreak', 'is_painttouch', 'is_fromturnover',
        # Free throws
        'freethrow_subtype',
        # Player identity
        'person_id', 'player_name', 'description',
    ]

    return df[keep].reset_index(drop=True)

# ── 5. Full pipeline ───────────────────────────────────────────────────────────
def run_pipeline(seasons, output_path='nba_win_prob_dataset.csv', batch_size=100):
    all_dfs = []
    done_ids = set()

    # Resume from existing file if present
    if os.path.exists(output_path):
        existing = pd.read_csv(output_path)
        done_ids = set(existing['game_id'].astype(str).unique())
        all_dfs.append(existing)
        print(f"Resuming — {len(done_ids)} games already processed\n")

    games_processed = 0
    games_skipped   = 0

    for season_year in seasons:
        for season_type, is_playoffs in [('regular', False), ('playoffs', True)]:
            label = f"{season_year}-{str(season_year+1)[-2:]} {'Playoffs' if is_playoffs else 'Regular Season'}"
            game_ids = generate_game_ids(season_year, season_type)
            game_ids = [g for g in game_ids if g not in done_ids]

            print(f"── {label}: trying {len(game_ids)} IDs ──")

            for game_id in tqdm(game_ids, desc=label):
                raw = try_get_pbp(game_id)

                if raw is None:
                    games_skipped += 1
                    time.sleep(0.3)
                    continue

                clean = build_game_state(raw, game_id, is_playoffs)
                if clean is not None:
                    all_dfs.append(clean)
                    done_ids.add(game_id)
                    games_processed += 1

                # Checkpoint save every batch_size real games
                if games_processed > 0 and games_processed % batch_size == 0:
                    pd.concat(all_dfs, ignore_index=True).to_csv(output_path, index=False)
                    print(f"\n  ✓ Checkpoint: {games_processed} games saved")

                time.sleep(0.6)

    # Final save
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        final_df.to_csv(output_path, index=False)
        print(f"\n✓ Done! {final_df['game_id'].nunique()} games | {len(final_df):,} rows → {output_path}")
        return final_df
    else:
        print("No data collected.")
        return None

# ── 6. Run ─────────────────────────────────────────────────────────────────────
seasons = [2022, 2023, 2024]  # 2022-23, 2023-24, 2024-25
df_full = run_pipeline(seasons, output_path='nba_win_prob_dataset.csv')

── 2022-23 Regular Season: trying 1230 IDs ──


2022-23 Regular Season:   8%|▊         | 99/1230 [02:24<26:43,  1.42s/it]


  ✓ Checkpoint: 100 games saved


2022-23 Regular Season:  16%|█▌        | 199/1230 [04:57<37:40,  2.19s/it]


  ✓ Checkpoint: 200 games saved


...

── 2024-25 Playoffs: trying 105 IDs ──


2024-25 Playoffs:  96%|█████████▌| 101/105 [01:35<00:04,  1.05s/it]


  ✓ Checkpoint: 3700 games saved


2024-25 Playoffs: 100%|██████████| 105/105 [02:15<00:00,  1.29s/it]



✓ Done! 3703 games | 1,598,131 rows → nba_win_prob_dataset.csv


In [21]:
print(df_full[df_full['is_playoffs']==1]['game_id'].unique())

<ArrowStringArray>
['0042300101', '0042300102', '0042300103', '0042300104', '0042300105',
 '0042400101', '0042400102', '0042400103', '0042400104', '0042500101',
 '0042500102', '0042500103', '0042500104', '0042500105']
Length: 14, dtype: str


In [22]:
# Check playoff game IDs
import pandas as pd
df = pd.read_csv('nba_win_prob_dataset.csv')
print(df[df['is_playoffs']==1]['game_id'].unique())

# Check player_name nulls — what actions are they?
print(df[df['player_name'].isna()]['actionType'].value_counts())

[42300101 42300102 42300103 42300104 42300105 42400101 42400102 42400103
 42400104 42500101 42500102 42500103 42500104 42500105]
actionType
rebound     61687
timeout     40939
turnover     5466
foul          550
ejection       34
Name: count, dtype: int64


In [23]:
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

# Playoff game IDs follow: 004YY0RSGGG
# YY = season year, R = round (1-4), S = series (01-08), GGG = game number (1-7)
# Let's test a few from different rounds/series of 2022-23

test_ids = [
    '0042200101',  # 2022 playoffs round 1 series 1 game 1
    '0042200201',  # round 2 series 1 game 1  
    '0042200301',  # conf finals series 1 game 1
    '0042200401',  # finals game 1
    '0042200111',  # round 1 series 1 game 1 — different format?
    '0042200121',  # 
]

for gid in test_ids:
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{gid}.json"
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(gid, r.status_code)

0042200101 200
0042200201 200
0042200301 200
0042200401 200
0042200111 200
0042200121 200


In [25]:
# Known working IDs from our status code test
working = ['0042200101', '0042200201', '0042200301', '0042200401', '0042200111', '0042200121']

# Break them down character by character
for gid in working:
    print(f"{gid} → 004 | {gid[3:5]} | {gid[5]} | {gid[6]} | {gid[7:9]} | {gid[9]}")

0042200101 → 004 | 22 | 0 | 0 | 10 | 1
0042200201 → 004 | 22 | 0 | 0 | 20 | 1
0042200301 → 004 | 22 | 0 | 0 | 30 | 1
0042200401 → 004 | 22 | 0 | 0 | 40 | 1
0042200111 → 004 | 22 | 0 | 0 | 11 | 1
0042200121 → 004 | 22 | 0 | 0 | 12 | 1


In [26]:
# Then test a few more to understand series/game encoding
test_ids = [
    '0042200102',  # same series game 2?
    '0042200112',  # different?
    '0042200211',  # round 2 series 1 game 1?
    '0042200311',  # conf finals?
    '0042200411',  # finals?
    '0042200108',  # series 1 round 1 game 8 (shouldnt exist)?
    '0042200102',  
    '0042200103',
    '0042200104',
    '0042200105',
    '0042200106',
    '0042200107',  # game 7?
    '0042200108',  # should 404
]

for gid in test_ids:
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{gid}.json"
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(gid, r.status_code)

0042200102 200
0042200112 200
0042200211 200
0042200311 200
0042200411 403
0042200108 403
0042200102 200
0042200103 200
0042200104 200
0042200105 200
0042200106 403
0042200107 403
0042200108 403


In [29]:
gid = '0042200101'
url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{gid}.json"
r = requests.get(url, headers=HEADERS, timeout=15)
data = r.json()
# Print everything except actions (too long)
game_info = {k: v for k, v in data['game'].items() if k != 'actions'}
print(game_info)

{'gameId': '0042200101'}


In [34]:
import requests, pandas as pd, time, os
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

def generate_playoff_ids(season_year):
    short_year = str(season_year + 1)[-2:]
    ids = []
    series_codes = [10,11,12,13,14,15,16,17, 20,21,22,23, 30,31, 40]
    for ss in series_codes:
        for game in range(1, 8):
            ids.append(f'004{short_year}00{ss}{game}')
    return ids

def run_playoff_patch(seasons, output_path='nba_win_prob_dataset.csv'):
    existing = pd.read_csv(output_path)
    done_ids = set(existing['game_id'].astype(str).unique())
    all_dfs = [existing]
    games_added = 0

    for season_year in seasons:
        short_year = str(season_year + 1)[-2:]
        game_ids = generate_playoff_ids(season_year)
        game_ids = [g for g in game_ids if g.lstrip('0') not in done_ids]
        print(f"\n── {season_year}-{short_year} Playoffs: {len(game_ids)} IDs to try ──")

        for game_id in tqdm(game_ids):
            raw = try_get_pbp(game_id)
            if raw is None:
                time.sleep(0.3)
                continue

            clean = build_game_state(raw, game_id, is_playoffs=True)
            if clean is not None:
                all_dfs.append(clean)
                done_ids.add(game_id.lstrip('0'))
                games_added += 1
                print(f"  ✓ {game_id}")

            if games_added > 0 and games_added % 25 == 0:
                pd.concat(all_dfs, ignore_index=True).to_csv(output_path, index=False)
                print(f"  Checkpoint: {games_added} games added")

            time.sleep(0.6)

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_csv(output_path, index=False)
    print(f"\n✓ Done! Added {games_added} playoff games")
    print(f"  Total: {final_df['game_id'].nunique()} games | {len(final_df):,} rows")
    return final_df

# 2022-23 and 2023-24 playoffs only (not current season)
seasons = [2022, 2023]
df_patched = run_playoff_patch(seasons)


── 2022-23 Playoffs: 100 IDs to try ──


  2%|▏         | 2/100 [00:02<01:43,  1.06s/it]

  ✓ 0042300111


  3%|▎         | 3/100 [00:03<02:15,  1.39s/it]

  ✓ 0042300112


...

  ✓ 0042400406


 99%|█████████▉| 100/101 [06:02<00:03,  3.17s/it]

  ✓ 0042400407


100%|██████████| 101/101 [06:04<00:00,  3.61s/it]



✓ Done! Added 157 playoff games
  Total: 3860 games | 1,664,320 rows


In [38]:
import requests, pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

# Check the first one
r1 = requests.get("https://cdn.nba.com/static/json/staticData/scheduleLeagueV2.json", headers=HEADERS, timeout=30)
data1 = r1.json()
print("Keys:", list(data1.keys()))
# Peek at structure
import json
print(json.dumps(data1, indent=2)[:2000])

Keys: ['meta', 'leagueSchedule']
{
  "meta": {
    "version": 1,
    "request": "http://nba.cloud/league/00/2025-26/scheduleleaguev2?Format=json",
    "time": "2026-05-21T15:49:50.495Z"
  },
  "leagueSchedule": {
    "seasonYear": "2025-26",
    "leagueId": "00",
    "gameDates": [
      {
        "gameDate": "10/02/2025 00:00:00",
        "games": [
          {
            "gameId": "0012500008",
            "gameCode": "20251002/PHINYK",
            "gameStatus": 3,
            "gameStatusText": "Final",
            "gameSequence": 1,
            "gameDateEst": "2025-10-02T00:00:00Z",
            "gameTimeEst": "1900-01-01T12:00:00Z",
            "gameDateTimeEst": "2025-10-02T12:00:00Z",
            "gameDateUTC": "2025-10-02T04:00:00Z",
            "gameTimeUTC": "1900-01-01T16:00:00Z",
            "gameDateTimeUTC": "2025-10-02T16:00:00Z",
            "awayTeamTime": "2025-10-02T12:00:00Z",
            "homeTeamTime": "2025-10-02T12:00:00Z",
            "day": "Thu",
            "

In [41]:
import requests, pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

def parse_schedule(year):
    url = f"https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/{year}/league/00_full_schedule.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    data = r.json()
    
    rows = []
    for month in data['lscd']:
        for game in month['mscd']['g']:
            # gcode format: 20230101/SACMEM — away=SAC, home=MEM
            gcode = game['gcode']
            date_str = gcode[:8]
            teams = gcode[9:]
            away_tricode = teams[:3]
            home_tricode = teams[3:]
            
            rows.append({
                'game_id': str(game['gid']).zfill(10),
                'game_date': pd.to_datetime(game['gdte']),
                'home_tricode': home_tricode,
                'away_tricode': away_tricode,
                'season_year': year,
            })
    
    return pd.DataFrame(rows)

# Pull all 3 seasons
schedules = []
for year in ['2022', '2023', '2024']:
    df_sched = parse_schedule(year)
    schedules.append(df_sched)
    print(f"{year}: {len(df_sched)} games, date range {df_sched['game_date'].min().date()} to {df_sched['game_date'].max().date()}")

schedule_df = pd.concat(schedules, ignore_index=True)
print(f"\nTotal: {len(schedule_df)} games")
print(schedule_df.head())

2022: 1394 games, date range 2022-09-30 to 2023-06-12
2023: 1396 games, date range 2023-10-05 to 2024-06-17
2024: 1400 games, date range 2024-10-04 to 2025-06-22

Total: 4190 games
      game_id  game_date home_tricode away_tricode season_year
0  0022200547 2023-01-01          MEM          SAC        2022
1  0022200548 2023-01-01          MIL          WAS        2022
2  0022200549 2023-01-01          DEN          BOS        2022
3  0022200550 2023-01-02          NYK          PHX        2022
4  0022200551 2023-01-02          CHA          LAL        2022


In [42]:
# Now compute rest days per team
# Each game appears once — we need to expand to one row per team per game
home = schedule_df[['game_id','game_date','home_tricode','season_year']].copy()
home.columns = ['game_id','game_date','tricode','season_year']
home['is_home'] = 1

away = schedule_df[['game_id','game_date','away_tricode','season_year']].copy()
away.columns = ['game_id','game_date','tricode','season_year']
away['is_home'] = 0

team_games = pd.concat([home, away], ignore_index=True).sort_values(['tricode','game_date'])

# Rest days = days since last game for this team
team_games['prev_game_date'] = team_games.groupby('tricode')['game_date'].shift(1)
team_games['rest_days'] = (team_games['game_date'] - team_games['prev_game_date']).dt.days
team_games['is_back_to_back'] = (team_games['rest_days'] == 1).astype(int)
team_games['is_3_in_4'] = (team_games['rest_days'] <= 2).astype(int)

print(team_games[['tricode','game_date','rest_days','is_back_to_back']].head(20))
print("\nRest days distribution:")
print(team_games['rest_days'].value_counts().sort_index().head(10))

     tricode  game_date  rest_days  is_back_to_back
4975     ADL 2022-10-02        NaN                0
4992     ADL 2022-10-06        4.0                0
800      ATL 2022-10-06        NaN                0
5003     ATL 2022-10-08        2.0                0
5019     ATL 2022-10-12        4.0                0
5034     ATL 2022-10-14        2.0                0
852      ATL 2022-10-19        5.0                0
867      ATL 2022-10-21        2.0                0
885      ATL 2022-10-23        2.0                0
5094     ATL 2022-10-26        3.0                0
5107     ATL 2022-10-28        2.0                0
5122     ATL 2022-10-29        1.0                1
5136     ATL 2022-10-31        2.0                0
5147     ATL 2022-11-02        2.0                0
981      ATL 2022-11-05        3.0                0
996      ATL 2022-11-07        2.0                0
1009     ATL 2022-11-09        2.0                0
1020     ATL 2022-11-10        1.0                1
5225     ATL

In [43]:
# Now join back to main dataset using home/away tricodes
# We need home_rest and away_rest per game_id
home_rest = team_games[team_games['is_home']==1][['game_id','rest_days','is_back_to_back','is_3_in_4','game_date']].copy()
home_rest.columns = ['game_id','home_rest_days','home_b2b','home_3in4','game_date']

away_rest = team_games[team_games['is_home']==0][['game_id','rest_days','is_back_to_back','is_3_in_4']].copy()
away_rest.columns = ['game_id','away_rest_days','away_b2b','away_3in4']

rest_df = home_rest.merge(away_rest, on='game_id')
print(rest_df.head(10))
print(f"\nTotal games with rest data: {len(rest_df)}")

# Save for joining later
rest_df.to_csv('game_rest_features.csv', index=False)
print("Saved to game_rest_features.csv")

      game_id  home_rest_days  home_b2b  home_3in4  game_date  away_rest_days  \
0  0012200023             NaN         0          0 2022-10-06             5.0   
1  0022200005             5.0         0          0 2022-10-19             5.0   
2  0022200020             2.0         0          1 2022-10-21             2.0   
3  0022200038             2.0         0          1 2022-10-23             2.0   
4  0022200134             3.0         0          0 2022-11-05             1.0   
5  0022200149             2.0         0          1 2022-11-07             2.0   
6  0022200162             2.0         0          1 2022-11-09             2.0   
7  0022200173             1.0         1          1 2022-11-10             3.0   
8  0022200214             2.0         0          1 2022-11-16             2.0   
9  0022200235             3.0         0          0 2022-11-19             3.0   

   away_b2b  away_3in4  
0         0          0  
1         0          0  
2         0          1  
3       

In [47]:
def compute_game_team_stats(df):
    rows = []
    
    for (game_id, tricode), grp in df.groupby(['game_id', 'teamTricode']):
        if pd.isna(tricode) or tricode == '':
            continue
        
        shots_2pt = grp[grp['actionType'] == '2pt']
        shots_3pt = grp[grp['actionType'] == '3pt']
        fts       = grp[grp['actionType'] == 'freethrow']
        turnovers = grp[grp['actionType'] == 'turnover']
        rebounds  = grp[grp['actionType'] == 'rebound']
        
        fgm_2 = (shots_2pt['shot_result'] == 'Made').sum()
        fga_2 = len(shots_2pt)
        fgm_3 = (shots_3pt['shot_result'] == 'Made').sum()
        fga_3 = len(shots_3pt)
        fgm   = fgm_2 + fgm_3
        fga   = fga_2 + fga_3
        ftm   = (fts['shot_result'] == 'Made').sum()
        fta   = len(fts)
        tov   = len(turnovers)
        oreb  = rebounds['description'].str.lower().str.contains('off:', na=False).sum()
        dreb  = rebounds['description'].str.lower().str.contains('def:', na=False).sum()
        pts   = fgm_2 * 2 + fgm_3 * 3 + ftm

        efg         = (fgm + 0.5 * fgm_3) / fga if fga > 0 else None
        possessions = fga + 0.44 * fta + tov
        tov_pct     = tov / possessions if possessions > 0 else None
        ft_rate     = fta / fga if fga > 0 else None

        rows.append({
            'game_id': game_id, 'tricode': tricode,
            'pts': pts, 'fgm': fgm, 'fga': fga,
            'fgm_3': fgm_3, 'fga_3': fga_3,
            'ftm': ftm, 'fta': fta, 'tov': tov,
            'oreb': oreb, 'dreb': dreb,
            'efg': efg, 'tov_pct': tov_pct,
            'ft_rate': ft_rate, 'possessions': possessions,
        })
    
    return pd.DataFrame(rows)

print("Computing per-game team stats... (this may take a minute)")
game_team_stats = compute_game_team_stats(df)
print(f"Shape: {game_team_stats.shape}")
print(game_team_stats.head(10).to_string())

Computing per-game team stats... (this may take a minute)
Shape: (7710, 16)
      game_id tricode  pts  fgm  fga  fgm_3  fga_3  ftm  fta  tov  oreb  dreb       efg   tov_pct   ft_rate  possessions
0  0022300001     CLE  116   44   84      8     28   20   25   13    35    35  0.571429  0.120370  0.297619       108.00
1  0022300001     IND  121   45   86     15     31   16   24   19    40    40  0.610465  0.164417  0.279070       115.56
2  0022300002     MIL  110   35   82     20     39   20   28   14    41    41  0.548780  0.129247  0.341463       108.32
3  0022300002     NYK  105   38   96     10     39   19   25   11    56    56  0.447917  0.093220  0.260417       118.00
4  0022300003     MIA  121   48   80     13     27   12   14   21    37    37  0.681250  0.195969  0.175000       107.16
5  0022300003     WAS  114   46   81     13     28    9   17   20    30    30  0.648148  0.184366  0.209877       108.48
6  0022300004     BKN  109   44   96     18     45    3    5   11    39    39

In [48]:
# ── Step 2: Join opponent stats for OREB% and ratings ────────────────────────
opp = game_team_stats[['game_id','tricode','pts','possessions','dreb','oreb']].copy()
opp.columns = ['game_id','opp_tricode','opp_pts','opp_possessions','opp_dreb','opp_oreb']

game_team_stats = game_team_stats.merge(opp, on='game_id', how='left')
game_team_stats = game_team_stats[game_team_stats['tricode'] != game_team_stats['opp_tricode']]

game_team_stats['oreb_pct'] = (
    game_team_stats['oreb'] /
    (game_team_stats['oreb'] + game_team_stats['opp_dreb']).replace(0, np.nan)
)
game_team_stats['off_rtg'] = (
    game_team_stats['pts'] / game_team_stats['possessions'] * 100
).replace([np.inf, -np.inf], np.nan)
game_team_stats['def_rtg'] = (
    game_team_stats['opp_pts'] / game_team_stats['opp_possessions'] * 100
).replace([np.inf, -np.inf], np.nan)
game_team_stats['net_rtg'] = game_team_stats['off_rtg'] - game_team_stats['def_rtg']
game_team_stats['won'] = (game_team_stats['pts'] > game_team_stats['opp_pts']).astype(int)

print(game_team_stats[['game_id','tricode','pts','opp_pts','efg','tov_pct',
                        'oreb_pct','off_rtg','def_rtg','net_rtg','won']].head(6).to_string())

       game_id tricode  pts  opp_pts       efg   tov_pct  oreb_pct     off_rtg     def_rtg    net_rtg  won
1   0022300001     CLE  116      121  0.571429  0.120370  0.466667  107.407407  104.707511   2.699896    0
2   0022300001     IND  121      116  0.610465  0.164417  0.533333  104.707511  107.407407  -2.699896    1
5   0022300002     MIL  110      105  0.548780  0.129247  0.422680  101.550960   88.983051  12.567909    1
6   0022300002     NYK  105      110  0.447917  0.093220  0.577320   88.983051  101.550960 -12.567909    0
9   0022300003     MIA  121      114  0.681250  0.195969  0.552239  112.915267  105.088496   7.826771    1
10  0022300003     WAS  114      121  0.648148  0.184366  0.447761  105.088496  112.915267  -7.826771    0


In [51]:
# Re-parse schedules to get team_dates
import requests, pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

def parse_schedule(year):
    url = f"https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/{year}/league/00_full_schedule.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    data = r.json()
    rows = []
    for month in data['lscd']:
        for game in month['mscd']['g']:
            gcode = game['gcode']
            rows.append({
                'game_id':      str(game['gid']).zfill(10),
                'game_date':    pd.to_datetime(game['gdte']),
                'home_tricode': gcode[9:][3:],
                'away_tricode': gcode[9:][:3],
            })
    return pd.DataFrame(rows)

schedule_df = pd.concat([parse_schedule(y) for y in ['2022','2023','2024']], ignore_index=True)

home_sched = schedule_df[['game_id','game_date','home_tricode']].rename(columns={'home_tricode':'tricode'})
away_sched = schedule_df[['game_id','game_date','away_tricode']].rename(columns={'away_tricode':'tricode'})
team_dates = pd.concat([home_sched, away_sched]).drop_duplicates()

# Also resave with tricodes so we don't need to do this again
schedule_df.to_csv('schedule_with_tricodes.csv', index=False)
print(f"team_dates: {len(team_dates)} rows")
print(team_dates.head())

team_dates: 8380 rows
      game_id  game_date tricode
0  0022200547 2023-01-01     MEM
1  0022200548 2023-01-01     MIL
2  0022200549 2023-01-01     DEN
3  0022200550 2023-01-02     NYK
4  0022200551 2023-01-02     CHA


In [52]:
# Now step 3
game_team_stats = game_team_stats.merge(team_dates, on=['game_id','tricode'], how='left')
game_team_stats = game_team_stats.sort_values(['tricode','game_date']).reset_index(drop=True)

matched = game_team_stats['game_date'].notna().sum()
print(f"Games with dates: {matched} / {len(game_team_stats)}")
print(game_team_stats[['tricode','game_date','off_rtg','def_rtg','net_rtg','won']].head(10).to_string())

Games with dates: 5250 / 7710
  tricode  game_date     off_rtg     def_rtg    net_rtg  won
0     ATL 2023-10-25   92.034806   99.622123  -7.587317    0
1     ATL 2023-10-27  105.078809  106.490872  -1.412063    0
2     ATL 2023-10-29  106.116310   93.378608  12.737702    1
3     ATL 2023-10-30  121.044605  102.169982  18.874623    1
4     ATL 2023-11-01  102.297765  102.403521  -0.105755    1
5     ATL 2023-11-04  105.525051  100.729087   4.795965    1
6     ATL 2023-11-06   89.258468  106.671182 -17.412714    0
7     ATL 2023-11-09  103.270224   95.659164   7.611060    1
8     ATL 2023-11-11   93.226138  106.363636 -13.137499    0
9     ATL 2023-11-14  112.059765  102.459016   9.600749    1


In [53]:
# ── Step 4: Rolling features at 10, 20, season windows ───────────────────────
stat_cols = ['efg','tov_pct','ft_rate','oreb_pct','off_rtg','def_rtg','net_rtg','won','pts']

def add_rolling(df, cols, window, label):
    for col in cols:
        df[f'{col}_last{label}'] = (
            df.groupby('tricode')[col]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )
    return df

print("Computing rolling features... (may take a moment)")
game_team_stats = add_rolling(game_team_stats, stat_cols, 10,  '10')
game_team_stats = add_rolling(game_team_stats, stat_cols, 20,  '20')
game_team_stats = add_rolling(game_team_stats, stat_cols, 82, 'season')

# Sanity check — OKC rolling net rating should be high
okc = game_team_stats[game_team_stats['tricode']=='OKC'].tail(5)
print(okc[['game_date','net_rtg','net_rtg_last10','net_rtg_last20','net_rtg_lastseason']].to_string())

game_team_stats.to_csv('team_rolling_stats.csv', index=False)
print(f"\nSaved — {len(game_team_stats)} rows, {len(game_team_stats.columns)} columns")

Computing rolling features... (may take a moment)
     game_date    net_rtg  net_rtg_last10  net_rtg_last20  net_rtg_lastseason
5469       NaT  19.718514       19.380431       13.802677           12.090994
5470       NaT -18.513015       18.664094       15.203754           12.459663
5471       NaT -21.210638       15.482647       13.740090           12.158867
5472       NaT  47.495652       11.000812       12.022080           11.795859
5473       NaT   1.503853       15.824139       14.106052           12.487789

Saved — 7710 rows, 54 columns


In [54]:
# What game IDs are NOT matching?
unmatched = game_team_stats[game_team_stats['game_date'].isna()]['game_id'].unique()
print(f"Unmatched game IDs: {len(unmatched)}")
print("Sample unmatched:", sorted(unmatched)[:20])
print()

# Are they all playoffs?
unmatched_series = pd.Series(unmatched).astype(str).str.zfill(10)
print("Prefixes of unmatched:")
print(unmatched_series.str[:4].value_counts())

Unmatched game IDs: 1230
Sample unmatched: ['0022500001', '0022500002', '0022500003', '0022500004', '0022500005', '0022500006', '0022500007', '0022500008', '0022500009', '0022500010', '0022500011', '0022500012', '0022500013', '0022500014', '0022500015', '0022500016', '0022500017', '0022500018', '0022500019', '0022500020']

Prefixes of unmatched:
0022    1230
Name: count, dtype: int64


In [55]:
# Fetch 2025-26 schedule from CDN (which we know works) — but we need 2024-25
# The CDN one returned 2025-26, so let's try year 2025 on data.nba.com
r = requests.get(
    "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2025/league/00_full_schedule.json",
    headers=HEADERS, timeout=30
)
print(r.status_code)
if r.status_code == 200:
    data = r.json()
    rows = []
    for month in data['lscd']:
        for game in month['mscd']['g']:
            gcode = game['gcode']
            rows.append({
                'game_id':      str(game['gid']).zfill(10),
                'game_date':    pd.to_datetime(game['gdte']),
                'home_tricode': gcode[9:][3:],
                'away_tricode': gcode[9:][:3],
            })
    sched_2025 = pd.DataFrame(rows)
    print(f"Games: {len(sched_2025)}, date range: {sched_2025['game_date'].min()} to {sched_2025['game_date'].max()}")
    print(sched_2025[sched_2025['game_id'].str.startswith('00225')].head())

200
Games: 1271, date range: 2025-10-02 00:00:00 to 2026-04-12 00:00:00
      game_id  game_date home_tricode away_tricode
0  0022500471 2026-01-01          BKN          HOU
1  0022500472 2026-01-01          DET          MIA
2  0022500473 2026-01-01          DAL          PHI
3  0022500474 2026-01-01          SAC          BOS
4  0022500475 2026-01-01          LAC          UTA


In [56]:
for season in ['2024-25', '2023-24', '2022-23']:
    url = f"https://cdn.nba.com/static/json/staticData/scheduleLeagueV2_{season}.json"
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(season, r.status_code)

2024-25 403
2023-24 403
2022-23 403


In [57]:
# ── Step 4: Rolling features — sort by game_id as proxy for chronological order
# game_ids are sequential so this is valid
game_team_stats = game_team_stats.sort_values(['tricode', 'game_id']).reset_index(drop=True)

stat_cols = ['efg','tov_pct','ft_rate','oreb_pct','off_rtg','def_rtg','net_rtg','won','pts']

def add_rolling(df, cols, window, label):
    for col in cols:
        df[f'{col}_last{label}'] = (
            df.groupby('tricode')[col]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )
    return df

print("Computing rolling features...")
game_team_stats = add_rolling(game_team_stats, stat_cols, 10,   '10')
game_team_stats = add_rolling(game_team_stats, stat_cols, 20,   '20')
game_team_stats = add_rolling(game_team_stats, stat_cols, 82, 'season')

# Sanity check
okc = game_team_stats[game_team_stats['tricode']=='OKC'].tail(5)
print(okc[['game_id','net_rtg','net_rtg_last10','net_rtg_last20','net_rtg_lastseason']].to_string())

game_team_stats.to_csv('team_rolling_stats.csv', index=False)
print(f"\nSaved — {len(game_team_stats)} rows, {len(game_team_stats.columns)} columns")

Computing rolling features...
         game_id    net_rtg  net_rtg_last10  net_rtg_last20  net_rtg_lastseason
5469  0042400403 -10.512348        8.980064        8.252238           10.354545
5470  0042400404   6.152275        6.219080        8.484335           10.025620
5471  0042400405   8.556008        7.362121        8.785422           10.330134
5472  0042400406  -9.242667        5.464149        7.097890           10.130425
5473  0042400407   8.764669        1.937980        6.050789            9.925397

Saved — 7710 rows, 54 columns


In [59]:
# We already know this works — fetch current season schedule
r = requests.get("https://cdn.nba.com/static/json/staticData/scheduleLeagueV2.json", headers=HEADERS, timeout=30)
data = r.json()

rows = []
for date_entry in data['leagueSchedule']['gameDates']:
    for game in date_entry['games']:
        gcode = game['gameCode']  # format: 20251002/PHINYK
        date_str = gcode[:8]
        teams = gcode[9:]
        rows.append({
            'game_id':      str(game['gameId']).zfill(10),
            'game_date':    pd.to_datetime(date_str, format='%Y%m%d'),
            'home_tricode': teams[3:],
            'away_tricode': teams[:3],
        })

current_season = pd.DataFrame(rows)
# Filter to only regular season (0022500xxx)
current_season = current_season[current_season['game_id'].str.startswith('002')]
print(f"Current season games: {len(current_season)}")
print(current_season.head())

Current season games: 1230
       game_id  game_date home_tricode away_tricode
71  0022500001 2025-10-21          OKC          HOU
72  0022500002 2025-10-21          LAL          GSW
73  0022500003 2025-10-22          NYK          CLE
74  0022500004 2025-10-22          DAL          SAS
75  0022500080 2025-10-22          CHA          BKN


In [60]:
# Build team_dates from current season and patch the gaps
home_c = current_season[['game_id','game_date','home_tricode']].rename(columns={'home_tricode':'tricode'})
away_c = current_season[['game_id','game_date','away_tricode']].rename(columns={'away_tricode':'tricode'})
current_team_dates = pd.concat([home_c, away_c]).drop_duplicates()

# Fill in missing dates in game_team_stats
game_team_stats = game_team_stats.merge(
    current_team_dates.rename(columns={'game_date':'game_date_fill'}),
    on=['game_id','tricode'],
    how='left'
)
game_team_stats['game_date'] = game_team_stats['game_date'].fillna(game_team_stats['game_date_fill'])
game_team_stats = game_team_stats.drop(columns=['game_date_fill'])

# Re-sort and recheck
game_team_stats = game_team_stats.sort_values(['tricode','game_date']).reset_index(drop=True)
still_missing = game_team_stats['game_date'].isna().sum()
print(f"Still missing dates: {still_missing}")
print(f"Matched: {game_team_stats['game_date'].notna().sum()} / {len(game_team_stats)}")

Still missing dates: 0
Matched: 7710 / 7710


In [61]:
# Re-run rolling features now that all dates are filled
print("Re-computing rolling features with complete dates...")
stat_cols = ['efg','tov_pct','ft_rate','oreb_pct','off_rtg','def_rtg','net_rtg','won','pts']

def add_rolling(df, cols, window, label):
    for col in cols:
        df[f'{col}_last{label}'] = (
            df.groupby('tricode')[col]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )
    return df

game_team_stats = add_rolling(game_team_stats, stat_cols, 10,  '10')
game_team_stats = add_rolling(game_team_stats, stat_cols, 20,  '20')
game_team_stats = add_rolling(game_team_stats, stat_cols, 82, 'season')

game_team_stats.to_csv('team_rolling_stats.csv', index=False)
print(f"Saved — {len(game_team_stats)} rows, {len(game_team_stats.columns)} columns")

Re-computing rolling features with complete dates...
Saved — 7710 rows, 54 columns


In [62]:
import pandas as pd
import numpy as np

# ── Load everything ───────────────────────────────────────────────────────────
df = pd.read_csv('nba_win_prob_dataset.csv')
df['game_id'] = df['game_id'].astype(str).str.zfill(10)
df = df[~df['game_id'].str.startswith('0042500')]

rest_df    = pd.read_csv('game_rest_features.csv', parse_dates=['game_date'])
team_stats = pd.read_csv('team_rolling_stats.csv')

# ── Build home and away feature sets from team_rolling_stats ─────────────────
rolling_cols = [c for c in team_stats.columns if any(
    c.endswith(s) for s in ['_last10','_last20','_lastseason']
)]
id_cols = ['game_id','tricode','off_rtg','def_rtg','net_rtg','efg',
           'tov_pct','ft_rate','oreb_pct','won','pts']
keep_cols = id_cols + rolling_cols

# We need to know which tricode is home vs away
# Use schedule to get home/away mapping
schedule = pd.read_csv('schedule_with_tricodes.csv')
home_map = schedule[['game_id','home_tricode','away_tricode']].copy()
home_map['game_id'] = home_map['game_id'].astype(str).str.zfill(10)

team_stats['game_id'] = team_stats['game_id'].astype(str).str.zfill(10)
team_stats_full = team_stats[keep_cols].merge(home_map, on='game_id', how='left')

# Split into home and away
home_stats = team_stats_full[team_stats_full['tricode'] == team_stats_full['home_tricode']].copy()
away_stats = team_stats_full[team_stats_full['tricode'] == team_stats_full['away_tricode']].copy()

# Rename columns with home_/away_ prefix
home_stats = home_stats.drop(columns=['tricode','home_tricode','away_tricode'])
away_stats = away_stats.drop(columns=['tricode','home_tricode','away_tricode'])

home_stats.columns = ['game_id'] + [f'home_{c}' for c in home_stats.columns if c != 'game_id']
away_stats.columns = ['game_id'] + [f'away_{c}' for c in away_stats.columns if c != 'game_id']

print(f"Home stats: {home_stats.shape}")
print(f"Away stats: {away_stats.shape}")
print(home_stats.head(3).to_string())

Home stats: (2625, 37)
Away stats: (2625, 37)
      game_id  home_off_rtg  home_def_rtg  home_net_rtg  home_efg  home_tov_pct  home_ft_rate  home_oreb_pct  home_won  home_pts  home_efg_last10  home_tov_pct_last10  home_ft_rate_last10  home_oreb_pct_last10  home_off_rtg_last10  home_def_rtg_last10  home_net_rtg_last10  home_won_last10  home_pts_last10  home_efg_last20  home_tov_pct_last20  home_ft_rate_last20  home_oreb_pct_last20  home_off_rtg_last20  home_def_rtg_last20  home_net_rtg_last20  home_won_last20  home_pts_last20  home_efg_lastseason  home_tov_pct_lastseason  home_ft_rate_lastseason  home_oreb_pct_lastseason  home_off_rtg_lastseason  home_def_rtg_lastseason  home_net_rtg_lastseason  home_won_lastseason  home_pts_lastseason
1  0022300079    105.078809    106.490872     -1.412063  0.551724      0.122592      0.344828       0.494382         0       120         0.446237             0.100402             0.354839              0.451613            92.034806            99.622123    

In [63]:
# ── Join everything to main dataset ──────────────────────────────────────────
# Start with main PBP dataset
df_final = df.copy()

# Join home team rolling stats
df_final = df_final.merge(home_stats, on='game_id', how='left')

# Join away team rolling stats  
df_final = df_final.merge(away_stats, on='game_id', how='left')

# Join rest features
rest_cols = ['game_id','home_rest_days','home_b2b','home_3in4',
             'away_rest_days','away_b2b','away_3in4','game_date']
rest_df['game_id'] = rest_df['game_id'].astype(str).str.zfill(10)
df_final = df_final.merge(rest_df[rest_cols], on='game_id', how='left')

print(f"Final shape: {df_final.shape}")
print(f"Columns: {len(df_final.columns)}")
print(f"Null counts (non-shot columns):")
core_cols = ['home_net_rtg_last10','away_net_rtg_last10',
             'home_rest_days','away_rest_days',
             'home_off_rtg_lastseason','away_def_rtg_lastseason']
print(df_final[core_cols].isnull().sum())

Final shape: (1661962, 117)
Columns: 117
Null counts (non-shot columns):
home_net_rtg_last10        545336
away_net_rtg_last10        545336
home_rest_days             538678
away_rest_days             538678
home_off_rtg_lastseason    545336
away_def_rtg_lastseason    545336
dtype: int64


In [64]:
# ── Add differential features (home minus away) ───────────────────────────────
# These are often more predictive than raw values
diff_base = ['off_rtg','def_rtg','net_rtg','efg','tov_pct','ft_rate','oreb_pct']
for window in ['last10','last20','lastseason']:
    for stat in diff_base:
        home_col = f'home_{stat}_{window}'
        away_col = f'away_{stat}_{window}'
        if home_col in df_final.columns and away_col in df_final.columns:
            df_final[f'diff_{stat}_{window}'] = df_final[home_col] - df_final[away_col]

# Game time of day (from game_date — afternoon vs evening games)
df_final['game_date'] = pd.to_datetime(df_final['game_date'])
df_final['day_of_week'] = df_final['game_date'].dt.dayofweek
df_final['is_weekend'] = df_final['day_of_week'].isin([4,5,6]).astype(int)

print(f"Final shape after differentials: {df_final.shape}")
print(f"Sample diff features:")
print(df_final[['game_id','diff_net_rtg_last10','diff_net_rtg_lastseason',
                'home_rest_days','away_rest_days']].drop_duplicates('game_id').head(10).to_string())

# Save
df_final.to_csv('nba_win_prob_final.csv', index=False)
print(f"\nSaved nba_win_prob_final.csv")
print(f"Shape: {df_final.shape}")
print(f"Columns: {df_final.columns.tolist()}")

Final shape after differentials: (1661962, 140)
Sample diff features:
         game_id  diff_net_rtg_last10  diff_net_rtg_lastseason  home_rest_days  away_rest_days
0     0022300001            -2.018877                -2.018877             2.0             2.0
430   0022300002             1.872412                 1.872412             2.0             2.0
883   0022300003             0.343560                 0.343560             2.0             2.0
1261  0022300004           -10.598751               -10.598751             2.0             2.0
1642  0022300005            -4.503433                -4.503433             2.0             2.0
2071  0022300006             1.051252                 1.051252             2.0             2.0
2501  0022300007             5.001999                 5.001999             2.0             2.0
2987  0022300008           -14.906727               -14.906727             2.0             2.0
3436  0022300009             0.884373                 0.884373             

In [67]:
import pandas as pd
import numpy as np
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://www.nba.com/",
}

# ── Rebuild full schedule with all 3 seasons including current ────────────────
def parse_schedule_old(year):
    url = f"https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/{year}/league/00_full_schedule.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    rows = []
    for month in r.json()['lscd']:
        for game in month['mscd']['g']:
            gcode = game['gcode']
            rows.append({
                'game_id':      str(game['gid']).zfill(10),
                'game_date':    pd.to_datetime(game['gdte']),
                'home_tricode': gcode[9:][3:],
                'away_tricode': gcode[9:][:3],
            })
    return pd.DataFrame(rows)

def parse_schedule_current():
    url = "https://cdn.nba.com/static/json/staticData/scheduleLeagueV2.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    rows = []
    for date_entry in r.json()['leagueSchedule']['gameDates']:
        for game in date_entry['games']:
            gcode = game['gameCode']
            date_str = gcode[:8]
            teams = gcode[9:]
            gid = str(game['gameId']).zfill(10)
            # Only regular season
            if not gid.startswith('002'):
                continue
            rows.append({
                'game_id':      gid,
                'game_date':    pd.to_datetime(date_str, format='%Y%m%d'),
                'home_tricode': teams[3:],
                'away_tricode': teams[:3],
            })
    return pd.DataFrame(rows)

print("Fetching schedules...")
sched_2022 = parse_schedule_old('2022')
sched_2023 = parse_schedule_old('2023')
sched_2024 = parse_schedule_old('2024')
sched_current = parse_schedule_current()

full_schedule = pd.concat([sched_2022, sched_2023, sched_2024, sched_current], ignore_index=True)
# Keep only regular season + playoffs we care about
full_schedule = full_schedule[full_schedule['game_id'].str.startswith(('002','004'))]
full_schedule.to_csv('schedule_with_tricodes.csv', index=False)
print(f"Full schedule: {len(full_schedule)} games")
print(full_schedule['game_id'].str[:4].value_counts())

Fetching schedules...
Full schedule: 5170 games
game_id
0022    4920
0042     250
Name: count, dtype: int64


In [68]:
# ── Re-run the full join with complete schedule ───────────────────────────────
df = pd.read_csv('nba_win_prob_dataset.csv', low_memory=False)
df['game_id'] = df['game_id'].astype(str).str.zfill(10)
df = df[~df['game_id'].str.startswith('0042500')]

team_stats = pd.read_csv('team_rolling_stats.csv')
team_stats['game_id'] = team_stats['game_id'].astype(str).str.zfill(10)

home_map = full_schedule[['game_id','home_tricode','away_tricode']].copy()
team_stats_full = team_stats.merge(home_map, on='game_id', how='left')

rolling_cols = [c for c in team_stats.columns if any(
    c.endswith(s) for s in ['_last10','_last20','_lastseason']
)]
id_cols = ['game_id','tricode','off_rtg','def_rtg','net_rtg','efg',
           'tov_pct','ft_rate','oreb_pct','won','pts']
keep_cols = id_cols + rolling_cols

team_stats_full = team_stats_full[keep_cols + ['home_tricode','away_tricode']]

home_stats = team_stats_full[team_stats_full['tricode'] == team_stats_full['home_tricode']].copy()
away_stats = team_stats_full[team_stats_full['tricode'] == team_stats_full['away_tricode']].copy()

home_stats = home_stats.drop(columns=['tricode','home_tricode','away_tricode'])
away_stats = away_stats.drop(columns=['tricode','home_tricode','away_tricode'])

home_stats.columns = ['game_id'] + [f'home_{c}' for c in home_stats.columns if c != 'game_id']
away_stats.columns = ['game_id'] + [f'away_{c}' for c in away_stats.columns if c != 'game_id']

print(f"Home stats: {home_stats.shape}, Away stats: {away_stats.shape}")
print(f"Nulls in home_map: {team_stats_full['home_tricode'].isna().sum()}")

Home stats: (3855, 37), Away stats: (3855, 37)
Nulls in home_map: 0


In [69]:
# ── Final join ────────────────────────────────────────────────────────────────
rest_df = pd.read_csv('game_rest_features.csv', parse_dates=['game_date'])
rest_df['game_id'] = rest_df['game_id'].astype(str).str.zfill(10)

df_final = df.copy()
df_final = df_final.merge(home_stats, on='game_id', how='left')
df_final = df_final.merge(away_stats, on='game_id', how='left')

rest_cols = ['game_id','home_rest_days','home_b2b','home_3in4',
             'away_rest_days','away_b2b','away_3in4','game_date']
df_final = df_final.merge(rest_df[rest_cols], on='game_id', how='left')

# Differential features
diff_base = ['off_rtg','def_rtg','net_rtg','efg','tov_pct','ft_rate','oreb_pct']
for window in ['last10','last20','lastseason']:
    for stat in diff_base:
        hc, ac = f'home_{stat}_{window}', f'away_{stat}_{window}'
        if hc in df_final.columns and ac in df_final.columns:
            df_final[f'diff_{stat}_{window}'] = df_final[hc] - df_final[ac]

df_final['game_date'] = pd.to_datetime(df_final['game_date'])
df_final['day_of_week'] = df_final['game_date'].dt.dayofweek
df_final['is_weekend']  = df_final['day_of_week'].isin([4,5,6]).astype(int)

# Final null check
print(f"\nFinal shape: {df_final.shape}")
null_check = df_final[['home_net_rtg_last10','away_net_rtg_last10',
                        'home_rest_days','away_rest_days']].isnull().sum()
print("Null counts:")
print(null_check)

df_final.to_csv('nba_win_prob_final.csv', index=False)
print("\nSaved nba_win_prob_final.csv")


Final shape: (1661962, 140)
Null counts:
home_net_rtg_last10      6658
away_net_rtg_last10      6658
home_rest_days         538678
away_rest_days         538678
dtype: int64

Saved nba_win_prob_final.csv


In [70]:
# ── Fix rest days — rebuild from full schedule ────────────────────────────────
full_schedule = pd.read_csv('schedule_with_tricodes.csv', parse_dates=['game_date'])

home = full_schedule[['game_id','game_date','home_tricode']].rename(columns={'home_tricode':'tricode'})
home['is_home'] = 1
away = full_schedule[['game_id','game_date','away_tricode']].rename(columns={'away_tricode':'tricode'})
away['is_home'] = 0

team_games = pd.concat([home, away]).sort_values(['tricode','game_date']).reset_index(drop=True)
team_games['prev_game_date'] = team_games.groupby('tricode')['game_date'].shift(1)
team_games['rest_days']      = (team_games['game_date'] - team_games['prev_game_date']).dt.days
team_games['is_back_to_back']= (team_games['rest_days'] == 1).astype(int)
team_games['is_3_in_4']      = (team_games['rest_days'] <= 2).astype(int)

home_rest = team_games[team_games['is_home']==1][['game_id','game_date','rest_days','is_back_to_back','is_3_in_4']].copy()
away_rest = team_games[team_games['is_home']==0][['game_id','rest_days','is_back_to_back','is_3_in_4']].copy()
home_rest.columns = ['game_id','game_date','home_rest_days','home_b2b','home_3in4']
away_rest.columns = ['game_id','away_rest_days','away_b2b','away_3in4']

rest_df_full = home_rest.merge(away_rest, on='game_id')
rest_df_full['game_id'] = rest_df_full['game_id'].astype(str).str.zfill(10)
rest_df_full.to_csv('game_rest_features.csv', index=False)

print(f"Rest features: {len(rest_df_full)} games")
print(f"Nulls: {rest_df_full[['home_rest_days','away_rest_days']].isnull().sum().to_dict()}")
print(rest_df_full.head())

Rest features: 5170 games
Nulls: {'home_rest_days': 14, 'away_rest_days': 16}
      game_id  game_date  home_rest_days  home_b2b  home_3in4  away_rest_days  \
0  0022200005 2022-10-19             NaN         0          0             NaN   
1  0022200020 2022-10-21             2.0         0          1             2.0   
2  0022200038 2022-10-23             2.0         0          1             2.0   
3  0022200134 2022-11-05             3.0         0          0             1.0   
4  0022200149 2022-11-07             2.0         0          1             2.0   

   away_b2b  away_3in4  
0         0          0  
1         0          1  
2         0          1  
3         1          1  
4         0          1  


In [71]:
# ── Re-join rest features and save final dataset ──────────────────────────────
df_final = pd.read_csv('nba_win_prob_final.csv', low_memory=False)
df_final['game_id'] = df_final['game_id'].astype(str).str.zfill(10)

# Drop old rest columns
df_final = df_final.drop(columns=['home_rest_days','home_b2b','home_3in4',
                                   'away_rest_days','away_b2b','away_3in4','game_date'], errors='ignore')

# Join fresh rest features
rest_cols = ['game_id','game_date','home_rest_days','home_b2b','home_3in4',
             'away_rest_days','away_b2b','away_3in4']
df_final = df_final.merge(rest_df_full[rest_cols], on='game_id', how='left')

df_final['game_date'] = pd.to_datetime(df_final['game_date'])
df_final['day_of_week'] = df_final['game_date'].dt.dayofweek
df_final['is_weekend']  = df_final['day_of_week'].isin([4,5,6]).astype(int)

# Final null check
print(f"Shape: {df_final.shape}")
null_check = df_final[['home_net_rtg_last10','away_net_rtg_last10',
                        'home_rest_days','away_rest_days']].isnull().sum()
print("Null counts:")
print(null_check)

# Check what games are still missing rest days
if df_final['home_rest_days'].isna().sum() > 0:
    missing = df_final[df_final['home_rest_days'].isna()]['game_id'].unique()
    print(f"\nGames missing rest: {len(missing)}")
    print(pd.Series(missing).str[:6].value_counts())

df_final.to_csv('nba_win_prob_final.csv', index=False)
print("\nSaved nba_win_prob_final.csv")

ParserError: Error tokenizing data. C error: out of memory

In [72]:
import pandas as pd

rest_df_full = pd.read_csv('game_rest_features.csv', parse_dates=['game_date'])
rest_df_full['game_id'] = rest_df_full['game_id'].astype(str).str.zfill(10)

# Process in chunks and write to new file
chunk_size = 100_000
output_path = 'nba_win_prob_final_v2.csv'
first_chunk = True

rest_cols = ['game_id','game_date','home_rest_days','home_b2b','home_3in4',
             'away_rest_days','away_b2b','away_3in4']

for chunk in pd.read_csv('nba_win_prob_final.csv', 
                          chunksize=chunk_size, 
                          low_memory=False):
    
    chunk['game_id'] = chunk['game_id'].astype(str).str.zfill(10)
    
    # Drop old rest/date columns
    chunk = chunk.drop(columns=['home_rest_days','home_b2b','home_3in4',
                                 'away_rest_days','away_b2b','away_3in4',
                                 'game_date','day_of_week','is_weekend'], 
                        errors='ignore')
    
    # Join fresh rest features
    chunk = chunk.merge(rest_df_full[rest_cols], on='game_id', how='left')
    
    # Re-add date features
    chunk['game_date'] = pd.to_datetime(chunk['game_date'])
    chunk['day_of_week'] = chunk['game_date'].dt.dayofweek
    chunk['is_weekend'] = chunk['day_of_week'].isin([4,5,6]).astype(int)
    
    chunk.to_csv(output_path, 
                 mode='w' if first_chunk else 'a',
                 header=first_chunk, 
                 index=False)
    first_chunk = False
    print('.', end='', flush=True)

print(f'\nDone! Saved to {output_path}')

# Quick validation
sample = pd.read_csv(output_path, nrows=1000)
print(f"Columns: {len(sample.columns)}")
print(f"Rest null check: {sample[['home_rest_days','away_rest_days']].isnull().sum().to_dict()}")

C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['day_of_week'] = chunk['game_date'].dt.dayofweek
C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['is_weekend'] = chunk['day_of_week'].isin([4,5,6]).astype(int)


.

C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['day_of_week'] = chunk['game_date'].dt.dayofweek
C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['is_weekend'] = chunk['day_of_week'].isin([4,5,6]).astype(int)


.

C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['day_of_week'] = chunk['game_date'].dt.dayofweek
C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['is_weekend'] = chunk['day_of_week'].isin([4,5,6]).astype(int)


...

.

C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['day_of_week'] = chunk['game_date'].dt.dayofweek
C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['is_weekend'] = chunk['day_of_week'].isin([4,5,6]).astype(int)


.

C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['day_of_week'] = chunk['game_date'].dt.dayofweek
C:\Users\joshu\AppData\Local\Temp\ipykernel_19480\1776170844.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk['is_weekend'] = chunk['day_of_week'].isin([4,5,6]).astype(int)


.
Done! Saved to nba_win_prob_final_v2.csv
Columns: 140
Rest null check: {'home_rest_days': 0, 'away_rest_days': 0}
